# 정수 인코딩(Integer Encoding)

자연어 처리에서는 각 단어에 고유한 정수 인덱스를 부여해 텍스트를 숫자 시퀀스로 변환합니다. 여기서는 빈도가 높은 15개 단어만 사전에 포함하고, 나머지는 OOV로 처리합니다.

## 1. NLTK 데이터 준비

In [19]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SJ\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\SJ\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SJ\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 2. 문장 토큰화와 전처리

In [20]:
from collections import Counter
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

raw_text = """The Little Prince, written by Antoine de Saint-Exupery, is a poetic tale about a young prince who travels from his home planet to Earth. The story begins with a pilot stranded in the Sahara Desert after his plane crashes. While trying to fix his plane, he meets a mysterious young boy, the Little Prince.

The Little Prince comes from a small asteroid called B-612, where he lives alone with a rose that he loves deeply. He recounts his journey to the pilot, describing his visits to several other planets. Each planet is inhabited by a different character, such as a king, a vain man, a drunkard, a businessman, a geographer, and a fox. Through these encounters, the Prince learns valuable lessons about love, responsibility, and the nature of adult behavior.

On Earth, the Little Prince meets various creatures, including a fox, who teaches him about relationships and the importance of taming, which means building ties with others. The fox's famous line, 'You become responsible, forever, for what you have tamed,' resonates with the Prince's feelings for his rose.

Ultimately, the Little Prince realizes that the essence of life is often invisible and can only be seen with the heart. After sharing his wisdom with the pilot, he prepares to return to his asteroid and his beloved rose. The story concludes with the pilot reflecting on the lessons learned from the Little Prince and the enduring impact of their friendship.

The narrative is a beautifully simple yet profound exploration of love, loss, and the importance of seeing beyond the surface of things."""

sentences = sent_tokenize(raw_text)
en_stopwords = set(stopwords.words('english'))
preprocessed_sentences = []

for sentence in sentences:
    tokens = word_tokenize(sentence.lower())
    tokens = [
        token for token in tokens
        if token.isalnum() and token not in en_stopwords and len(token) > 2
    ]
    preprocessed_sentences.append(tokens)  # 문장별 토큰 리스트를 저장

vocab = Counter(
    token
    for sentence in preprocessed_sentences
    for token in sentence
)

print(preprocessed_sentences[:3])
print(f'전체 단어 종류: {len(vocab)}')

[['little', 'prince', 'written', 'antoine', 'poetic', 'tale', 'young', 'prince', 'travels', 'home', 'planet', 'earth'], ['story', 'begins', 'pilot', 'stranded', 'sahara', 'desert', 'plane', 'crashes'], ['trying', 'fix', 'plane', 'meets', 'mysterious', 'young', 'boy', 'little', 'prince']]
전체 단어 종류: 107


## 3. 빈도순 단어 사전 생성

In [21]:
vocab_size = 15
vocab_sorted = vocab.most_common()

# 0은 패딩용으로 비워 두고, 단어 인덱스는 1부터 시작
word_to_idx = {
    word: index
    for index, (word, count) in enumerate(vocab_sorted[:vocab_size], start=1)
}

# 셀을 반복 실행해도 OOV 인덱스가 바뀌지 않음
word_to_idx['OOV'] = vocab_size + 1
idx_to_word = {index: word for word, index in word_to_idx.items()}

word_to_idx

{'prince': 1,
 'little': 2,
 'pilot': 3,
 'rose': 4,
 'fox': 5,
 'young': 6,
 'planet': 7,
 'earth': 8,
 'story': 9,
 'plane': 10,
 'meets': 11,
 'asteroid': 12,
 'lessons': 13,
 'love': 14,
 'importance': 15,
 'OOV': 16}

## 4. 정수 인코딩

사전에 없는 단어는 모두 OOV 인덱스로 변환합니다.

In [22]:
# 전처리 문장을 정수 인덱스 시퀀스로 인코딩(OOV 처리)
oov_idx = word_to_idx['OOV']  
encoded_sentences = [ 
    [word_to_idx.get(token, oov_idx) for token in sentence]
    for sentence in preprocessed_sentences
]

for tokens, encoded_sentence in zip(preprocessed_sentences, encoded_sentences):
    print(tokens)
    print(encoded_sentence)
    print('---')

encoded_sentences

['little', 'prince', 'written', 'antoine', 'poetic', 'tale', 'young', 'prince', 'travels', 'home', 'planet', 'earth']
[2, 1, 16, 16, 16, 16, 6, 1, 16, 16, 7, 8]
---
['story', 'begins', 'pilot', 'stranded', 'sahara', 'desert', 'plane', 'crashes']
[9, 16, 3, 16, 16, 16, 10, 16]
---
['trying', 'fix', 'plane', 'meets', 'mysterious', 'young', 'boy', 'little', 'prince']
[16, 16, 10, 11, 16, 6, 16, 2, 1]
---
['little', 'prince', 'comes', 'small', 'asteroid', 'called', 'lives', 'alone', 'rose', 'loves', 'deeply']
[2, 1, 16, 16, 12, 16, 16, 16, 4, 16, 16]
---
['recounts', 'journey', 'pilot', 'describing', 'visits', 'several', 'planets']
[16, 16, 3, 16, 16, 16, 16]
---
['planet', 'inhabited', 'different', 'character', 'king', 'vain', 'man', 'drunkard', 'businessman', 'geographer', 'fox']
[7, 16, 16, 16, 16, 16, 16, 16, 16, 16, 5]
---
['encounters', 'prince', 'learns', 'valuable', 'lessons', 'love', 'responsibility', 'nature', 'adult', 'behavior']
[16, 1, 16, 16, 13, 14, 16, 16, 16, 16]
---
['ear

[[2, 1, 16, 16, 16, 16, 6, 1, 16, 16, 7, 8],
 [9, 16, 3, 16, 16, 16, 10, 16],
 [16, 16, 10, 11, 16, 6, 16, 2, 1],
 [2, 1, 16, 16, 12, 16, 16, 16, 4, 16, 16],
 [16, 16, 3, 16, 16, 16, 16],
 [7, 16, 16, 16, 16, 16, 16, 16, 16, 16, 5],
 [16, 1, 16, 16, 13, 14, 16, 16, 16, 16],
 [8, 2, 1, 11, 16, 16, 16, 5, 16, 16, 15, 16, 16, 16, 16, 16],
 [5, 16, 16, 16, 16, 16, 16, 16, 1, 16, 4],
 [16, 2, 1, 16, 16, 16, 16, 16, 16, 16],
 [16, 16, 3, 16, 16, 12, 16, 4],
 [9, 16, 3, 16, 13, 16, 2, 1, 16, 16, 16],
 [16, 16, 16, 16, 16, 16, 14, 16, 15, 16, 16, 16, 16]]

In [27]:
from tensorflow.keras.preprocessing.text import Tokenizer

# 상위 15개만 사용 (필터링용), 그 외 토큰은 OOV로 처리
tokenizer = Tokenizer(num_words = 15, oov_token ='<OOV>')

tokenizer.fit_on_texts(preprocessed_sentences)
tokenizer.word_index  # 단어 -> 인덱스 딕셔너리

{'<OOV>': 1,
 'prince': 2,
 'little': 3,
 'pilot': 4,
 'rose': 5,
 'fox': 6,
 'young': 7,
 'planet': 8,
 'earth': 9,
 'story': 10,
 'plane': 11,
 'meets': 12,
 'asteroid': 13,
 'lessons': 14,
 'love': 15,
 'importance': 16,
 'written': 17,
 'antoine': 18,
 'poetic': 19,
 'tale': 20,
 'travels': 21,
 'home': 22,
 'begins': 23,
 'stranded': 24,
 'sahara': 25,
 'desert': 26,
 'crashes': 27,
 'trying': 28,
 'fix': 29,
 'mysterious': 30,
 'boy': 31,
 'comes': 32,
 'small': 33,
 'called': 34,
 'lives': 35,
 'alone': 36,
 'loves': 37,
 'deeply': 38,
 'recounts': 39,
 'journey': 40,
 'describing': 41,
 'visits': 42,
 'several': 43,
 'planets': 44,
 'inhabited': 45,
 'different': 46,
 'character': 47,
 'king': 48,
 'vain': 49,
 'man': 50,
 'drunkard': 51,
 'businessman': 52,
 'geographer': 53,
 'encounters': 54,
 'learns': 55,
 'valuable': 56,
 'responsibility': 57,
 'nature': 58,
 'adult': 59,
 'behavior': 60,
 'various': 61,
 'creatures': 62,
 'including': 63,
 'teaches': 64,
 'relationships'

In [28]:
tokenizer.index_word

{1: '<OOV>',
 2: 'prince',
 3: 'little',
 4: 'pilot',
 5: 'rose',
 6: 'fox',
 7: 'young',
 8: 'planet',
 9: 'earth',
 10: 'story',
 11: 'plane',
 12: 'meets',
 13: 'asteroid',
 14: 'lessons',
 15: 'love',
 16: 'importance',
 17: 'written',
 18: 'antoine',
 19: 'poetic',
 20: 'tale',
 21: 'travels',
 22: 'home',
 23: 'begins',
 24: 'stranded',
 25: 'sahara',
 26: 'desert',
 27: 'crashes',
 28: 'trying',
 29: 'fix',
 30: 'mysterious',
 31: 'boy',
 32: 'comes',
 33: 'small',
 34: 'called',
 35: 'lives',
 36: 'alone',
 37: 'loves',
 38: 'deeply',
 39: 'recounts',
 40: 'journey',
 41: 'describing',
 42: 'visits',
 43: 'several',
 44: 'planets',
 45: 'inhabited',
 46: 'different',
 47: 'character',
 48: 'king',
 49: 'vain',
 50: 'man',
 51: 'drunkard',
 52: 'businessman',
 53: 'geographer',
 54: 'encounters',
 55: 'learns',
 56: 'valuable',
 57: 'responsibility',
 58: 'nature',
 59: 'adult',
 60: 'behavior',
 61: 'various',
 62: 'creatures',
 63: 'including',
 64: 'teaches',
 65: 'relationsh

In [29]:
tokenizer.word_counts # 전체 빈도수

OrderedDict([('little', 6),
             ('prince', 9),
             ('written', 1),
             ('antoine', 1),
             ('poetic', 1),
             ('tale', 1),
             ('young', 2),
             ('travels', 1),
             ('home', 1),
             ('planet', 2),
             ('earth', 2),
             ('story', 2),
             ('begins', 1),
             ('pilot', 4),
             ('stranded', 1),
             ('sahara', 1),
             ('desert', 1),
             ('plane', 2),
             ('crashes', 1),
             ('trying', 1),
             ('fix', 1),
             ('meets', 2),
             ('mysterious', 1),
             ('boy', 1),
             ('comes', 1),
             ('small', 1),
             ('asteroid', 2),
             ('called', 1),
             ('lives', 1),
             ('alone', 1),
             ('rose', 3),
             ('loves', 1),
             ('deeply', 1),
             ('recounts', 1),
             ('journey', 1),
             ('describing', 

In [30]:
# 문장(토큰 리스트)을 단어 인덱스로 치환하여 정수형 시퀀스 리스트로 변환
sequences = tokenizer.texts_to_sequences(preprocessed_sentences)
sequences

[[3, 2, 1, 1, 1, 1, 7, 2, 1, 1, 8, 9],
 [10, 1, 4, 1, 1, 1, 11, 1],
 [1, 1, 11, 12, 1, 7, 1, 3, 2],
 [3, 2, 1, 1, 13, 1, 1, 1, 5, 1, 1],
 [1, 1, 4, 1, 1, 1, 1],
 [8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6],
 [1, 2, 1, 1, 14, 1, 1, 1, 1, 1],
 [9, 3, 2, 12, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1],
 [6, 1, 1, 1, 1, 1, 1, 1, 2, 1, 5],
 [1, 3, 2, 1, 1, 1, 1, 1, 1, 1],
 [1, 1, 4, 1, 1, 13, 1, 5],
 [10, 1, 4, 1, 14, 1, 3, 2, 1, 1, 1],
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]

- num_words 옵션을 안주면 모든 단어가 그대로 인덱싱되어 OOV가 거의 없음
    - 단, fit_on_texts 사전 생성 이후 새로 등장한 단어/데이터는 OOV로 가게 된다.
- num_words를 쓰면
    - 실제 모델 학습에서 임베딩 테이블 크기(=파라미터/메모리/속도) 관리
    - 희귀 단어를 다 포함하게 되면 노이즈 + 차원이 커져 -> 일반화에 불리
    - 그래서 보통 상위 N개 단어 사용후 나머지 OOV 처리
    - 작은 데이터 : 1000~5000
    - 간단한 분류 : 5000~20000
    - 데이터가 매우 클 경우 : 20000개 ~ 50000

In [32]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 샘플 데이터
corpus = ["자연어 처리는 재미있다.", "자연어 처리는 어렵다.", "데이터는 유용하다."]

# Bag-of-Words
vectorizer = CountVectorizer()             # BoW 벡터라이저 생성
bow = vectorizer.fit_transform(corpus)     # 단어사전 학습(생성) + BoW 벡터로 변환(희소행렬)
print("Bag-of-Words:\\n", bow.toarray())   # 밀집 배열 형태로 변환해서 출력



# TF-IDF
tfidf = TfidfVectorizer()                   # TF-IDF 벡터라이저 생성
tfidf_matrix = tfidf.fit_transform(corpus)  # 단어사전 학습(쌩성) + TF-IDF 벡터로 변환(희소행렬)
print("TF-IDF:\\n", tfidf_matrix.toarray()) # TF-IDF 결과를 배열로 변환해서 출력

Bag-of-Words:\n [[0 0 0 1 1 1]
 [0 1 0 1 0 1]
 [1 0 1 0 0 0]]
TF-IDF:\n [[0.         0.         0.         0.51785612 0.68091856 0.51785612]
 [0.         0.68091856 0.         0.51785612 0.         0.51785612]
 [0.70710678 0.         0.70710678 0.         0.         0.        ]]


In [33]:
# 샘플 텍스트
text = "자연어 처리는 재미있다. 자연어는 어렵지만 재미있다."

# 단어 빈도 계산
from collections import Counter
words = text.split()
word_count = Counter(words)

# 문장 길이 계산
sentences = text.split(". ")
sentence_lengths = [len(sentence.split()) for sentence in sentences]

print("단어 빈도:", word_count)
print("문장 길이:", sentence_lengths)

단어 빈도: Counter({'재미있다.': 2, '자연어': 1, '처리는': 1, '자연어는': 1, '어렵지만': 1})
문장 길이: [3, 3]


In [34]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 임의의 단어 벡터
word1 = np.array([[1, 2, 3]])
word2 = np.array([[2, 3, 4]])

# 코사인 유사도 계산
similarity = cosine_similarity(word1, word2)
print("코사인 유사도:", similarity[0][0])

코사인 유사도: 0.9925833339709303


In [35]:
from konlpy.tag import Okt

# 텍스트 데이터
text = "자연어 처리는 재미있다."

# 형태소 분석
okt = Okt()
morphs = okt.morphs(text)
print("형태소 분석 결과:", morphs)

형태소 분석 결과: ['자연어', '처리', '는', '재미있다', '.']


In [36]:
from sklearn.feature_extraction.text import CountVectorizer

# 문장 데이터
corpus = ["자연어 처리는 재미있다.", "Python으로 가능하다."]

# BoW 벡터화
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
print("BoW 벡터화 결과:\\n", X.toarray())
print("단어 사전:", vectorizer.vocabulary_)

BoW 벡터화 결과:\n [[0 0 1 1 1]
 [1 1 0 0 0]]
단어 사전: {'자연어': 2, '처리는': 4, '재미있다': 3, 'python으로': 0, '가능하다': 1}


In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 문장 데이터
corpus = ["자연어 처리는 재미있다.", "Python으로 가능하다."]

# TF-IDF 벡터화
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)
print("TF-IDF 결과:\\n", X.toarray())
print("단어 사전:", vectorizer.vocabulary_)

TF-IDF 결과:\n [[0.         0.         0.57735027 0.57735027 0.57735027]
 [0.70710678 0.70710678 0.         0.         0.        ]]
단어 사전: {'자연어': 2, '처리는': 4, '재미있다': 3, 'python으로': 0, '가능하다': 1}


In [41]:
from gensim.models import Word2Vec

# 샘플 문장
sentences = [["자연어", "처리", "재미있다"], ["Python", "가능하다"]]

# Word2Vec 모델 학습
model = Word2Vec(
    sentences,        # 토큰화된 문장 (2차원)
    vector_size=100,  # 단어 벡터 차원 수 (100차원)
    window=3,         # 주변 문맥으로 볼 단어 : 좌우 3개
    min_count=1,      # 최소 등장 횟수 1 이상인 단어만 사전에 포함
    sg=0)             # 0 = CBOW(주변 -> 중심 단어 예측), 1 = Skip-gram(중심 -> 주변 단어 예측)

# 단어 벡터 확인 : 학습된 '자연어' 임베딩 벡터(길이)
print("단어 '자연어'의 벡터:", model.wv["자연어"])

단어 '자연어'의 벡터: [-0.00713902  0.00124103 -0.00717672 -0.00224462  0.0037193   0.00583312
  0.00119818  0.00210273 -0.00411039  0.00722533 -0.00630704  0.00464722
 -0.00821997  0.00203647 -0.00497705 -0.00424769 -0.00310898  0.00565521
  0.0057984  -0.00497465  0.00077333 -0.00849578  0.00780981  0.00925729
 -0.00274233  0.00080022  0.00074665  0.00547788 -0.00860608  0.00058446
  0.00686942  0.00223159  0.00112468 -0.00932216  0.00848237 -0.00626413
 -0.00299237  0.00349379 -0.00077263  0.00141129  0.00178199 -0.0068289
 -0.00972481  0.00904058  0.00619805 -0.00691293  0.00340348  0.00020606
  0.00475375 -0.00711994  0.00402695  0.00434743  0.00995737 -0.00447374
 -0.00138926 -0.00731732 -0.00969783 -0.00908026 -0.00102275 -0.00650329
  0.00484973 -0.00616403  0.00251919  0.00073944 -0.00339215 -0.00097922
  0.00997913  0.00914589 -0.00446183  0.00908303 -0.00564176  0.00593092
 -0.00309722  0.00343175  0.00301723  0.00690046 -0.00237388  0.00877504
  0.00758943 -0.00954765 -0.00800821 -

In [43]:
model.wv.similarity("자연어", "처리") # 단어간 유사도 측정 (코사인 유사도)

np.float32(-0.0446171)

In [44]:
model.wv.most_similar("자연어", topn = 5) # 가장 유사한 단어

[('재미있다', 0.17018885910511017),
 ('Python', 0.004503022879362106),
 ('가능하다', -0.027750348672270775),
 ('처리', -0.04461711645126343)]

In [45]:
import gensim.downloader as api

# 1) 미리 학습된 GloVe 로드 (처음 1번은 다운로드 시간이 걸림)
# 50차원 버전: 가볍고 실습용으로 좋음
glove = api.load("glove-wiki-gigaword-50")

# 2) 단어 벡터 확인 (길이=50)
print("king 벡터 차원:", glove["king"].shape)

# 3) 코사인 유사도
print("similarity(king, queen):", glove.similarity("king", "queen"))
print("similarity(king, banana):", glove.similarity("king", "banana"))

# 4) 가장 비슷한 단어 top-n
print("most_similar('king'):", glove.most_similar("king", topn=5))

# 5) 벡터 연산(관계) 예시: king - man + woman ≈ queen
print(glove.most_similar(positive=["king", "woman"], negative=["man"], topn=5))

[==================================================] 100.0% 66.0/66.0MB downloaded
king 벡터 차원: (50,)
similarity(king, queen): 0.7839043
similarity(king, banana): 0.2207408
most_similar('king'): [('prince', 0.8236179351806641), ('queen', 0.7839043140411377), ('ii', 0.7746230363845825), ('emperor', 0.7736247777938843), ('son', 0.766719400882721)]
[('queen', 0.8523604273796082), ('throne', 0.7664334177970886), ('prince', 0.7592144012451172), ('daughter', 0.7473883628845215), ('elizabeth', 0.7460219860076904)]


In [46]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# 1) 예시 문서(문장) 준비
docs = [
    "자연어 처리는 재미있다",
    "파이썬으로 자연어 처리를 할 수 있다",
    "축구 경기는 정말 재미있다",
    "오늘 날씨가 좋다"
]

# 2) Doc2Vec은 문서마다 태그(아이디)가 필요함
tagged_docs = [
    TaggedDocument(words=d.split(), tags=[f"DOC_{i}"])
    for i, d in enumerate(docs)
]

# 3) Doc2Vec 모델 학습
model = Doc2Vec(
    documents=tagged_docs, # 태그가 붙은 문서로 학습
    vector_size=50,   # 문서 벡터 차원
    window=3,         # 문맥 범위
    min_count=1,      # 최소 등장 횟수
    workers=2,        # CPU 쓰레드
    epochs=50         # 학습 반복
)

# 4) 학습된 문서 벡터 확인 (첫 번재 문서의 벡터 상위 10개)
print("DOC_0 벡터 일부:", model.dv["DOC_0"][:10])

# 5) 특정 문서와 가장 유사한 문서 찾기
print("\nDOC_0와 유사한 문서:")
print(model.dv.most_similar("DOC_0", topn=3)) # 새로운 문장 문서 벡터로 추정

# 6) 새 문장(학습에 없던 문장)도 벡터로 추정 가능(infer_vector)
new_doc = "자연어 처리는 파이썬으로 가능하다"
new_vec = model.infer_vector(new_doc.split())

print("\n새 문장과 유사한 문서:")
print(model.dv.most_similar([new_vec], topn=3))

DOC_0 벡터 일부: [-0.01058207 -0.01189603 -0.01985634  0.01694183  0.00701691  0.00072557
 -0.01981407 -0.01017613 -0.01967002  0.00412226]

DOC_0와 유사한 문서:
[('DOC_1', 0.2878270447254181), ('DOC_2', 0.1307842880487442), ('DOC_3', -0.07168306410312653)]

새 문장과 유사한 문서:
[('DOC_0', 0.2532918453216553), ('DOC_1', 0.2054271548986435), ('DOC_2', 0.14233563840389252)]


In [48]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# 문장 데이터
corpus = ["자연어 처리는 재미있다.", "Python으로 가능하다."]

# BoW 벡터화
vectorizer = CountVectorizer()         # BoW 빈도수 기반 벡터라이저
X = vectorizer.fit_transform(corpus)   # 단어사전 학습(생성) + 희소행렬 생성

# LDA 모델 학습
lda = LatentDirichletAllocation(
    n_components=2,                     # 찾을 주제 개수
    random_state=0                      # 재현성
)
lda.fit(X)                              # BoW행렬로 LDA 학습 (주제-단어 분포 / 문서 - 주제 분포 추정)

# 주제별 단어 분포
print("주제별 단어 분포:\\n", lda.components_) # 각 주제에서 단어가 얼마나 중요한지 (n_topics, n_words)

주제별 단어 분포:\n [[1.4913572  1.4913572  0.50945773 0.50945773 0.50945773]
 [0.5086428  0.5086428  1.49054227 1.49054227 1.49054227]]


In [50]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()   # Vectorizer 단어 사전(인덱스 -> 단어 배열)

for topic_idx, topic in enumerate(lda.components_):
    top_idx = topic.argsort()[::-1][:5]  # 가중치가 큰 단어 인덱스 (내림차순 후 상위 5개 )
    print(f"Topic {topic_idx}: {[(feature_names[i], topic) for i in top_idx]}")

Topic 0: [('python으로', array([1.4913572 , 1.4913572 , 0.50945773, 0.50945773, 0.50945773])), ('가능하다', array([1.4913572 , 1.4913572 , 0.50945773, 0.50945773, 0.50945773])), ('자연어', array([1.4913572 , 1.4913572 , 0.50945773, 0.50945773, 0.50945773])), ('처리는', array([1.4913572 , 1.4913572 , 0.50945773, 0.50945773, 0.50945773])), ('재미있다', array([1.4913572 , 1.4913572 , 0.50945773, 0.50945773, 0.50945773]))]
Topic 1: [('재미있다', array([0.5086428 , 0.5086428 , 1.49054227, 1.49054227, 1.49054227])), ('처리는', array([0.5086428 , 0.5086428 , 1.49054227, 1.49054227, 1.49054227])), ('자연어', array([0.5086428 , 0.5086428 , 1.49054227, 1.49054227, 1.49054227])), ('가능하다', array([0.5086428 , 0.5086428 , 1.49054227, 1.49054227, 1.49054227])), ('python으로', array([0.5086428 , 0.5086428 , 1.49054227, 1.49054227, 1.49054227]))]
